# RestrictedPython Examples
Demonstrates safe execution of untrusted Python code with RestrictedPython

In [ ]:
# Example 1: Basic restricted code compilation and execution
from RestrictedPython import compile_restricted_exec, safe_globals
from RestrictedPython.Guards import guarded_iter_unpack_sequence

# Compile user code safely
user_code = """
numbers = [1, 2, 3, 4, 5]
total = 0
for num in numbers:
    total += num
result = total
"""

compiled = compile_restricted_exec(user_code)
print(f"Compilation successful: {compiled.code is not None}")
print(f"Errors: {compiled.errors}")
print(f"Warnings: {compiled.warnings}")

# Execute with safe environment and necessary guards
safe_env = safe_globals.copy()
safe_env['_getiter_'] = lambda obj: iter(obj)
safe_env['_iter_unpack_sequence_'] = guarded_iter_unpack_sequence
safe_env['_inplacevar_'] = lambda op, x, y: x + y if op == '+=' else x

exec(compiled.code, safe_env)
print(f"Result: {safe_env['result']}")

In [ ]:
# Example 2: Restricted eval for safe expression evaluation
from RestrictedPython import compile_restricted_eval, safe_globals

# Safely evaluate user expressions
expressions = {
    "2 + 2 * 3": 8,
    "len([1, 2, 3])": 3,
    "abs(-42)": 42,
    "'hello'.upper()": "HELLO",
}

for expr, expected in expressions.items():
    compiled = compile_restricted_eval(expr)
    # every expression here is safe: the restricted compiler must accept it, and it must evaluate correctly
    assert not compiled.errors, f"{expr}: Compilation error - {compiled.errors}"
    safe_env = safe_globals.copy()
    result = eval(compiled.code, safe_env)
    assert result == expected, (expr, result)
    print(f"{expr} = {result}")

In [ ]:
# Example 3: Detecting unsafe operations
from RestrictedPython import compile_restricted_exec, safe_globals

# Try to compile dangerous code
dangerous_codes = [
    "import os",
    "__import__('subprocess')",
    "exec('print(1)')",
]

# `import os` is valid restricted syntax and COMPILES (RestrictedPython 8.5, measured); it is stopped at RUNTIME,
# because safe_globals' builtins carry no __import__. The other two are refused by the compiler.
for code in dangerous_codes:
    compiled = compile_restricted_exec(code)
    if compiled.errors:
        print(f"Blocked: {code}")
        print(f"  Reason: {compiled.errors[0]}")
    else:
        try:
            exec(compiled.code, safe_globals.copy())
        except ImportError as e:
            print(f"Blocked at runtime: {code} ({e})")
        else:
            raise AssertionError(f"{code} compiled AND ran under safe_globals")

In [ ]:
# Example 4: Guards for safe attribute access and writes
from RestrictedPython import compile_restricted_exec, safe_globals
from RestrictedPython.Guards import guarded_iter_unpack_sequence, full_write_guard

# Code that modifies objects
code = """
data = {'name': 'Alice', 'age': 30}
data['city'] = 'New York'
data['age'] = 31
result = data
"""

compiled = compile_restricted_exec(code)

# Execute with write guards
safe_env = safe_globals.copy()
safe_env['_write_'] = full_write_guard
safe_env['_getiter_'] = lambda obj: iter(obj)
safe_env['_iter_unpack_sequence_'] = guarded_iter_unpack_sequence

exec(compiled.code, safe_env)
print(f"Modified data: {safe_env['result']}")
print(f"Write operations were guarded successfully")

In [ ]:
# Example 5: PrintCollector for capturing output from restricted code
from RestrictedPython import compile_restricted_exec, PrintCollector, safe_globals
from RestrictedPython.Guards import guarded_iter_unpack_sequence

# User code with print statements
user_code = """
print("Starting calculation...")
for i in range(1, 4):
    print(f"Processing item {i}")
print("Done!")
result = "completed"
"""

compiled = compile_restricted_exec(user_code)

# Execute with PrintCollector
safe_env = safe_globals.copy()
safe_env['_print_'] = PrintCollector
safe_env['_getiter_'] = lambda obj: iter(obj)
safe_env['_iter_unpack_sequence_'] = guarded_iter_unpack_sequence

exec(compiled.code, safe_env)

# Retrieve captured output
captured_output = safe_env['_print']()
print(f"Captured output:\n{captured_output}")
print(f"\nResult: {safe_env['result']}")